# 04 — Target Size Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shelly-serafimovich/target-aware-infrared-sr/blob/main/notebooks/04_target_size_analysis.ipynb)

Focused analysis of thermal-drone bounding-box sizes. The goal is to quantify how many targets are genuinely tiny, especially after the ×4 downsampling used by the SR experiment.

This notebook uses the corrected YOLO annotations and reports **target instances**, not images. No SR metric or detector result is evaluated here.


## 1. Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_REPO = 'https://github.com/shelly-serafimovich/target-aware-infrared-sr.git'
project_dir = Path('/content/target-aware-infrared-sr')
if not project_dir.exists():
    subprocess.run(['git','clone',PROJECT_REPO,str(project_dir)],check=True)
else:
    subprocess.run(['git','pull','--ff-only'],cwd=project_dir,check=True)

DATASET_ROOT = Path('/content/drive/MyDrive/thermal-drone-dataset')
ANN_ROOT = project_dir / 'annotations' / 'corrected_yolo'
SCALE = 4
splits = ['valid']  # Match the frozen DifIISR validation experiment
print('Annotations:', ANN_ROOT)


## 2. Build one row per target

YOLO boxes are normalized. We recover pixel width/height using the source image dimensions. `max_dim_lr` is the largest bbox dimension after ideal ×4 spatial downsampling.


In [ ]:
import zipfile

LOCAL_IMAGES = Path('/content/target_size_images')
LOCAL_IMAGES.mkdir(exist_ok=True)
exts = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}

def image_dir_for_split(split):
    src = DATASET_ROOT / split
    direct = [p for p in src.iterdir() if p.is_file() and p.suffix.lower() in exts]
    if direct:
        return src
    zips = list(src.glob('*.zip'))
    if len(zips) != 1:
        raise FileNotFoundError(f'Expected one ZIP for {split}, found {len(zips)}')
    out = LOCAL_IMAGES / split
    marker = out / '.complete'
    if not marker.exists():
        if out.exists():
            import shutil; shutil.rmtree(out)
        out.mkdir(parents=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(out)
        marker.touch()
    candidates = [out] + [p for p in out.rglob('*') if p.is_dir()]
    for p in candidates:
        if any(x.is_file() and x.suffix.lower() in exts for x in p.iterdir()):
            return p
    raise FileNotFoundError(f'No images found for {split}')

rows=[]
for split in splits:
    img_dir=image_dir_for_split(split)
    image_map={p.stem:p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in exts}
    label_dir=ANN_ROOT/split/'labels'
    for lp in sorted(label_dir.glob('*.txt')):
        lines=[x.strip() for x in lp.read_text().splitlines() if x.strip()]
        if not lines:
            continue
        img_path=image_map.get(lp.stem)
        if img_path is None:
            raise FileNotFoundError(f'Image missing for {split}/{lp.stem}')
        with Image.open(img_path) as im:
            W,H=im.size
        for target_idx,line in enumerate(lines):
            p=line.split()
            if len(p)<5:
                continue
            _,xc,yc,bw,bh=map(float,p[:5])
            w=bw*W; h=bh*H
            rows.append({'split':split,'image':lp.stem,'target_idx':target_idx,
                         'image_w':W,'image_h':H,'width_hr':w,'height_hr':h,
                         'max_dim_hr':max(w,h),'diagonal_hr':np.hypot(w,h),'area_hr':w*h,
                         'width_lr':w/SCALE,'height_lr':h/SCALE,
                         'max_dim_lr':max(w,h)/SCALE,'diagonal_lr':np.hypot(w,h)/SCALE,
                         'area_lr':(w*h)/(SCALE**2)})

df=pd.DataFrame(rows)
print(f'Target instances: {len(df):,}')
display(df.head())


## 3. How many targets are tiny?

We report explicit thresholds instead of relying on a vague definition of “small”. The main variable is `max_dim_lr`: the larger side of the target bbox in the LR image.


In [ ]:
bins=[0,2,4,8,16,np.inf]
labels=['≤2 px','2–4 px','4–8 px','8–16 px','>16 px']
df['lr_size_bin']=pd.cut(df['max_dim_lr'],bins=bins,labels=labels,include_lowest=True,right=True)

counts=df['lr_size_bin'].value_counts(sort=False)
table=pd.DataFrame({'targets':counts,'percent':100*counts/len(df)})
display(table.round({'percent':2}))

for t in [2,4,8,16]:
    n=int((df.max_dim_lr<=t).sum())
    print(f'max_dim_lr ≤ {t:>2}px: {n:>5,} / {len(df):,} = {100*n/len(df):6.2f}%')


## 4. Distribution by split

This checks whether the tiny-target population is concentrated in one split or is represented across train/validation/test.


In [ ]:
split_table=pd.crosstab(df['split'],df['lr_size_bin'])
split_pct=100*split_table.div(split_table.sum(axis=1),axis=0)
display(split_table)
display(split_pct.round(2))


## 5. Distribution plots


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df['max_dim_lr'],bins=50)
plt.xlabel('Target max bbox dimension in LR (pixels)')
plt.ylabel('Number of target instances')
plt.title('Thermal-drone target size after ×4 downsampling')
plt.xlim(0, min(40, df['max_dim_lr'].quantile(0.99)))
plt.show()

plt.figure(figsize=(8,5))
table['percent'].plot(kind='bar')
plt.xlabel('LR target-size bin')
plt.ylabel('Targets (%)')
plt.title('Target-size composition of the dataset')
plt.xticks(rotation=0)
plt.show()


## 6. Compact summary

Use these values when deciding whether statements such as “there are almost no small drones” are supported by the corrected dataset.


In [ ]:
summary={
    'n_targets':int(len(df)),
    'median_max_dim_hr':float(df.max_dim_hr.median()),
    'median_max_dim_lr':float(df.max_dim_lr.median()),
    'pct_lr_le_2':float(100*(df.max_dim_lr<=2).mean()),
    'pct_lr_le_4':float(100*(df.max_dim_lr<=4).mean()),
    'pct_lr_le_8':float(100*(df.max_dim_lr<=8).mean()),
    'pct_lr_le_16':float(100*(df.max_dim_lr<=16).mean()),
}
for k,v in summary.items():
    print(f'{k}: {v}')


## 7. Pixel-area buckets and 9×9 window coverage

Here we bucket targets by **LR bounding-box area in pixels**. We also report two interpretations of a 9×9 window: (1) bbox area ≤81 px², and (2) the stricter/geometrically correct condition for fitting fully inside a 9×9 window: width ≤9 px **and** height ≤9 px.


In [ ]:
# Integer pixel-area buckets (ceil so a fractional bbox is not understated)
df['area_lr_px_ceil'] = np.ceil(df['area_lr']).astype(int)

area_counts = (df.groupby('area_lr_px_ceil')
                 .agg(targets=('image','size'), images=('image','nunique'))
                 .reset_index()
                 .rename(columns={'area_lr_px_ceil':'pixel_area_bucket'}))
area_counts['target_percent'] = 100 * area_counts['targets'] / len(df)
display(area_counts)

# 9×9 checks
by_area = df['area_lr'] <= 81
fits_9x9 = (df['width_lr'] <= 9) & (df['height_lr'] <= 9)

def report(mask, name):
    sub = df[mask]
    print(f'{name}:')
    print(f'  targets: {len(sub):,} / {len(df):,} = {100*len(sub)/len(df):.2f}%')
    print(f'  unique images: {sub.image.nunique():,} / {df.image.nunique():,} = {100*sub.image.nunique()/df.image.nunique():.2f}%')

report(by_area, 'LR bbox area ≤ 81 px²')
report(fits_9x9, 'LR bbox fits fully inside 9×9 (width≤9 AND height≤9)')


## 8. Visual examples from each LR target-size group

Validation only. For each target-size bucket, show the **actual frozen degraded LR image** used by the DifIISR baseline next to its corresponding HR source image. The HR bbox is drawn on HR; the same normalized bbox is mapped to the actual LR image for visual comparison.


In [ ]:
from matplotlib.patches import Rectangle

N_EXAMPLES = 6
LR_VALID_DIR = DATASET_ROOT / 'difiisr_baseline' / 'valid' / 'LR_difiisr_train_degradation_v2'
assert LR_VALID_DIR.exists(), LR_VALID_DIR

hr_dir = image_dir_for_split('valid')
hr_map = {p.stem:p for p in hr_dir.iterdir() if p.is_file() and p.suffix.lower() in exts}
lr_files = [p for p in LR_VALID_DIR.rglob('*') if p.is_file() and p.suffix.lower() in exts]
lr_map = {p.stem:p for p in lr_files}
print(f'HR validation images: {len(hr_map):,}')
print(f'Actual frozen LR images: {len(lr_map):,}')

def get_norm_box(row):
    lp = ANN_ROOT / 'valid' / 'labels' / f"{row['image']}.txt"
    lines = [x.strip() for x in lp.read_text().splitlines() if x.strip()]
    p = lines[int(row['target_idx'])].split()
    _, xc, yc, bw, bh = map(float, p[:5])
    return xc, yc, bw, bh

def xywh_pixels(norm_box, W, H):
    xc, yc, bw, bh = norm_box
    return (xc-bw/2)*W, (yc-bh/2)*H, bw*W, bh*H

for group in labels:
    group_df = df[df['lr_size_bin'] == group]
    available = group_df[group_df['image'].isin(lr_map)]
    n = min(N_EXAMPLES, len(available))
    print(f'\n{group}: {len(group_df):,} validation targets — showing {n}')
    if n == 0:
        continue
    sample = available.sample(n=n, random_state=12345)
    fig, axes = plt.subplots(n, 2, figsize=(10, 4*n), squeeze=False)
    for r, (_, row) in enumerate(sample.iterrows()):
        norm_box = get_norm_box(row)
        hr = Image.open(hr_map[row['image']]).convert('L')
        lr = Image.open(lr_map[row['image']]).convert('L')
        for ax, img, tag in [(axes[r,0], hr, 'HR'), (axes[r,1], lr, 'Actual LR v2')]:
            W,H = img.size
            x,y,w,h = xywh_pixels(norm_box,W,H)
            ax.imshow(img,cmap='gray')
            ax.add_patch(Rectangle((x,y),w,h,fill=False,linewidth=2))
            ax.set_title(f"{tag} | bbox {w:.1f}×{h:.1f}px")
            ax.axis('off')
    fig.suptitle(f'{group} — validation only', y=1.01)
    plt.tight_layout()
    plt.show()


## 9. SCR and calibrated Hoyer on the actual LR

Validation only. This analysis uses the **actual frozen LR images** from the DifIISR experiment and focuses only on tiny targets with **LR max bbox dimension ≤ 8 px** (`≤2`, `2–4`, `4–8`).

The goal is simple: test whether Tal's SCR and calibrated Hoyer metrics distinguish tiny drone regions from **background-only regions directly in LR space**. No bicubic upsampling is used here.


In [ ]:
import cv2
TINY_LABELS=['≤2 px','2–4 px','4–8 px']
tiny_df=df[(df.max_dim_lr<=8)&df.image.isin(lr_map)].copy()
print(f'Tiny targets in frozen LR: {len(tiny_df):,}')
display(tiny_df.lr_size_bin.value_counts(sort=False).loc[TINY_LABELS].to_frame('targets'))

# Tal's 61/31 geometry was defined after x4 upsampling. In native LR we use
# scaled odd windows: 15x15 outer patch and 7x7 core.
OUTER=15; CORE=7
def core_ring(img,cx,cy,outer=OUTER,core=CORE):
    H,W=img.shape; cx=int(round(cx)); cy=int(round(cy)); ho=outer//2; hc=core//2
    x0,x1=cx-ho,cx+ho+1; y0,y1=cy-ho,cy+ho+1
    if x0<0 or y0<0 or x1>W or y1>H:return None,None
    p=img[y0:y1,x0:x1].astype(np.float32); c0=ho-hc; c1=c0+core
    c=p[c0:c1,c0:c1]; m=np.ones_like(p,dtype=bool); m[c0:c1,c0:c1]=False
    return c,p[m]
def calibrated_hoyer(core,ring):
    x=np.maximum(0.,core.astype(np.float32).ravel()-float(ring.mean())); rs=float(ring.std())
    l1=float(np.abs(x).sum()); l2=float(np.linalg.norm(x)); n=x.size
    if l2<=0:return 0.
    raw=(np.sqrt(n)-l1/l2)/(np.sqrt(n)-1.)
    expected=np.sqrt(n/2.)*rs; excess=max(0.,l2-expected)
    return float(raw*excess/(excess+expected+1e-6))
# Collect target patches first. SCR noise floor is the 5th percentile of positive LR ring std.
target_rows=[]
for _,r in tiny_df.iterrows():
    img=cv2.imread(str(lr_map[r.image]),cv2.IMREAD_GRAYSCALE)
    if img is None:continue
    H,W=img.shape; xc,yc,_,_=get_norm_box(r); c,ring=core_ring(img,xc*W,yc*H)
    if c is None:continue
    target_rows.append({'kind':'target','image':r.image,'target_idx':int(r.target_idx),'lr_size_bin':str(r.lr_size_bin),'core_max':float(c.max()),'bg_mean':float(ring.mean()),'bg_std':float(ring.std()),'hoyer':calibrated_hoyer(c,ring)})
targets_m=pd.DataFrame(target_rows)
positive=targets_m.loc[targets_m.bg_std>0,'bg_std'].to_numpy()
NOISE_FLOOR=float(np.percentile(positive,5)) if len(positive) else 1e-8
targets_m['scr']=(targets_m.core_max-targets_m.bg_mean)/np.maximum(targets_m.bg_std,NOISE_FLOOR)
print('LR SCR noise floor:',NOISE_FLOOR)
display(targets_m.head())


In [ ]:
# Build matched background patches from validation images with empty corrected labels.
rng=np.random.default_rng(12345)
bg_stems=[]
label_dir=ANN_ROOT/'valid'/'labels'
for stem in lr_map:
    lp=label_dir/f'{stem}.txt'
    if (not lp.exists()) or (not lp.read_text().strip()): bg_stems.append(stem)
bg_rows=[]
for stem in bg_stems:
    img=cv2.imread(str(lr_map[stem]),cv2.IMREAD_GRAYSCALE)
    if img is None or img.shape[0]<OUTER or img.shape[1]<OUTER:continue
    H,W=img.shape; ho=OUTER//2
    cx=int(rng.integers(ho,W-ho)); cy=int(rng.integers(ho,H-ho))
    c,ring=core_ring(img,cx,cy)
    bg_rows.append({'kind':'background','image':stem,'target_idx':-1,'lr_size_bin':'background','core_max':float(c.max()),'bg_mean':float(ring.mean()),'bg_std':float(ring.std()),'hoyer':calibrated_hoyer(c,ring)})
background_m=pd.DataFrame(bg_rows)
background_m['scr']=(background_m.core_max-background_m.bg_mean)/np.maximum(background_m.bg_std,NOISE_FLOOR)
metrics_lr=pd.concat([targets_m,background_m],ignore_index=True)
print(f'Target patches: {len(targets_m):,} | Background patches: {len(background_m):,}')
summary=(metrics_lr.groupby(['kind','lr_size_bin']).agg(n=('scr','size'),scr_median=('scr','median'),scr_mean=('scr','mean'),hoyer_median=('hoyer','median'),hoyer_mean=('hoyer','mean')).reset_index())
display(summary.round(4))


In [ ]:
# Distributions: tiny-target bins versus background.
order=TINY_LABELS+['background']
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for metric,ax in zip(['scr','hoyer'],axes):
    data=[metrics_lr.loc[metrics_lr.lr_size_bin==g,metric].dropna().to_numpy() for g in order]
    ax.boxplot(data,labels=order,showfliers=False)
    ax.set_title(('SCR' if metric=='scr' else 'Calibrated Hoyer')+' on actual frozen LR')
    ax.set_ylabel(metric.upper() if metric=='scr' else 'Hoyer')
    ax.set_xlabel('LR max bbox size')
plt.tight_layout(); plt.show()
print('Question to answer: do the tiny-target distributions separate from background, especially for ≤2 px and 2–4 px?')


### 9.1 AUROC: can each LR metric rank tiny targets above background?

Boxplots show distribution shifts, but they do not quantify separation. Here we compute **AUROC** separately for `≤2 px`, `2–4 px`, and `4–8 px`, always against the same background-patch set. An AUROC of 0.5 means chance-level ranking; larger values mean the metric more often assigns a higher score to a target patch than to a background patch.

This is a descriptive validation of the metric on frozen LR — **not** detector performance and not yet evidence that the metric will work as diffusion guidance.


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

auc_rows=[]
bg=metrics_lr[metrics_lr.kind=='background']
for size_bin in TINY_LABELS:
    tg=metrics_lr[(metrics_lr.kind=='target') & (metrics_lr.lr_size_bin==size_bin)]
    for metric in ['scr','hoyer']:
        y=np.r_[np.ones(len(tg),dtype=int),np.zeros(len(bg),dtype=int)]
        scores=np.r_[tg[metric].to_numpy(),bg[metric].to_numpy()]
        ok=np.isfinite(scores)
        auc=roc_auc_score(y[ok],scores[ok])
        auc_rows.append({'lr_size_bin':size_bin,'metric':metric.upper() if metric=='scr' else 'Calibrated Hoyer','n_targets':len(tg),'n_background':len(bg),'AUROC':auc})
auc_df=pd.DataFrame(auc_rows)
display(auc_df.pivot(index='lr_size_bin',columns='metric',values='AUROC').loc[TINY_LABELS].round(4))
display(auc_df.round(4))


In [ ]:
# ROC curves, kept separate by metric for readability.
for metric,title in [('scr','SCR'),('hoyer','Calibrated Hoyer')]:
    plt.figure(figsize=(6,5))
    for size_bin in TINY_LABELS:
        tg=metrics_lr[(metrics_lr.kind=='target') & (metrics_lr.lr_size_bin==size_bin)]
        y=np.r_[np.ones(len(tg),dtype=int),np.zeros(len(bg),dtype=int)]
        scores=np.r_[tg[metric].to_numpy(),bg[metric].to_numpy()]
        ok=np.isfinite(scores); fpr,tpr,_=roc_curve(y[ok],scores[ok]); auc=roc_auc_score(y[ok],scores[ok])
        plt.plot(fpr,tpr,label=f'{size_bin} (AUC={auc:.3f})')
    plt.plot([0,1],[0,1],'--',linewidth=1,label='Chance')
    plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
    plt.title(f'{title}: tiny targets vs background on frozen LR')
    plt.legend(); plt.grid(alpha=.2); plt.show()
